# PointDiT in motion, Kaggle evaluation

Scores saved PointDiT predictions on the TUM RGB-D subset and runs the follow-up analyses selected by the flags below. Runs on Kaggle CPU (Accelerator set to None) with Internet on.

## Setup

| Item | Kind | Notes |
|---|---|---|
| PointDiT TUM RGB-D Subset | dataset | always attached |
| Runner output with `predictions/<run name>/run.json` | notebook output | the predictions to score |
| Evaluation output with `eval/<run name>/per_frame.csv` | notebook output | needed when `RUN_EVALUATION` is False |
| `GITHUB_TOKEN` | Kaggle secret | only while the repo is private |
| `PREDICTIONS_DIR` | setting | None finds the single attached `run.json` |
| `PER_FRAME_CSV_OVERRIDE` | setting | exact `per_frame.csv` path instead of discovery |
| `WORKERS` | setting | `evaluate.py` workers, about 3.5 GB RAM each |
| `RENDER_SEQUENCES` | setting | sequences to render, None means all |
| `RENDER_INPUT` | setting | attached render output, None uses this session's |
| `STORY_MODES`, `STORY_ARGS` | setting | `render_story.py` modes and per mode options |
| `WORLD_MODES`, `WORLD_ARGS` | setting | `render_world_video.py` modes and options |
| `FUSION_ALL_TARGETS_CONFIG` | setting | one fusion config to rerun on all 300 targets |
| `FUSION_CONFIG_GROUP`, `FUSION_CACHE_INPUT`, `FUSION_SEQUENCES` | setting | set in the fusion cell |

No GPU, DINOv3 weights or `HF_TOKEN` are needed. Input discovery ignores any path containing `/Pointdit-In-Motion/`, so a project checkout inside an attached output is never picked up.

## Run flags

All flags default to False. The previews and frame checks always run for a single run.

| Flag | What it runs | Typical runtime |
|---|---|---|
| `RUN_EVALUATION` | `evaluate.py`, then `compare_eval_csv.py` against an attached older `per_frame.csv` | about 1.5 h |
| `RUN_FOCAL_SPLIT` | `focal_reliability.py`, writing `focal_split.csv` | a few minutes |
| `RUN_CONSISTENCY` | `consistency_eval.py` over all frame pairs at 33, 100 and 333 ms | about 2 to 3 h |
| `RUN_PERSISTENCE` | `persistence_eval.py` over the same pairs | about 1 to 1.5 h |
| `RUN_FUSION` | `fusion_prepare.py` and `fusion_eval.py` at oracle scale | about 2 to 3 h, edge group about 1 h |
| `RUN_FUSION_JOINT` | `fusion_joint.py` with jointly solved scale and oracle poses | about 1.5 to 2 h |
| `RUN_RENDER_PREPARE` | `fusion_render_prepare.py`, raw and fused depth per frame | not recorded |
| `RUN_RENDER_VIDEO` | `render_video.py`, then `export_fused_map.py` and `render_orbit.py` | not recorded |
| `RENDER_FLYTHROUGH` | an extra fused map flythrough with `RUN_RENDER_VIDEO` | not recorded |
| `RENDER_STORY` | `render_story.py` clips and a slowed side by side orbit | not recorded |
| `RENDER_WORLD` | `render_world_video.py` world frame videos | about 4 to 6 min per mode and sequence |
| `RUN_SAMPLING_EVAL` | sampler initialisation comparison, only alone | about 3 h |
| `RUN_SWEEP_EVAL` | `evaluate.py` and `sweep_accuracy.py` on every successful sweep run | about 2 h |

`RUN_SAMPLING_EVAL` needs every other single-run flag False. With `RUN_SWEEP_EVAL` the single-run steps are skipped unless `PREDICTIONS_DIR` is set.

In [ ]:
PROJECT_REPO = "https://github.com/Sainava/Pointdit-In-Motion.git"
PROJECT_BRANCH = "main"
# None uses the single run.json under /kaggle/input; set a path if several are attached.
PREDICTIONS_DIR = None
RUN_EVALUATION = False
RUN_FOCAL_SPLIT = False
RUN_CONSISTENCY = False
RUN_PERSISTENCE = False
RUN_FUSION = False
# Needs the aligned depth cache from RUN_FUSION or FUSION_CACHE_INPUT.
RUN_FUSION_JOINT = False
# Needs the aligned depth cache from RUN_FUSION or FUSION_CACHE_INPUT.
RUN_RENDER_PREPARE = False
# None renders every sequence of the cache.
RENDER_SEQUENCES = ["rgbd_dataset_freiburg1_360", "rgbd_dataset_freiburg1_desk"]
RUN_RENDER_VIDEO = False
# None uses this session's output; else a glob for one output root or its fusion/render/<run name>/.
RENDER_INPUT = None
RENDER_FLYTHROUGH = False
RENDER_STORY = False
STORY_MODES = ["hero", "flicker", "toggle", "stillflip", "cloud", "cloud_moving"]
# Every mode takes --fps, --start and --max-frames.
STORY_ARGS = {"toggle": ["--start", 20, "--max-frames", 80], "cloud_moving": ["--start", 20, "--max-frames", 120]}
RENDER_WORLD = False
WORLD_MODES = ["accumulate", "sliding"]
# For example ["--with-gt"] adds a third GT panel.
WORLD_ARGS = []
# None discovers the single per_frame.csv under /kaggle/input.
PER_FRAME_CSV_OVERRIDE = None
# Rerun this fusion config on all 300 targets per sequence, or None.
FUSION_ALL_TARGETS_CONFIG = None
# Each worker needs ~3.5 GB of RAM; torch already uses every core for one frame.
WORKERS = 1
RUN_SAMPLING_EVAL = False
if RUN_SAMPLING_EVAL and (RUN_EVALUATION or RUN_FOCAL_SPLIT or RUN_CONSISTENCY or RUN_PERSISTENCE or RUN_FUSION
                          or RUN_FUSION_JOINT or RUN_RENDER_PREPARE or RUN_RENDER_VIDEO or RENDER_STORY
                          or RENDER_WORLD):
    raise ValueError("RUN_SAMPLING_EVAL runs on its own: set RUN_EVALUATION, RUN_FOCAL_SPLIT, RUN_CONSISTENCY, "
                     "RUN_PERSISTENCE, RUN_FUSION, RUN_FUSION_JOINT, RUN_RENDER_PREPARE, RUN_RENDER_VIDEO, "
                     "RENDER_STORY and RENDER_WORLD to False")
if FUSION_ALL_TARGETS_CONFIG and not RUN_FUSION:
    raise ValueError("FUSION_ALL_TARGETS_CONFIG needs RUN_FUSION = True")
# The sweep attaches several run.json, so single-run steps then need PREDICTIONS_DIR.
RUN_SWEEP_EVAL = False
SINGLE_RUN = not RUN_SAMPLING_EVAL and (not RUN_SWEEP_EVAL or PREDICTIONS_DIR is not None)
if RUN_SWEEP_EVAL and not SINGLE_RUN and (RUN_EVALUATION or RUN_FOCAL_SPLIT or RUN_CONSISTENCY
                                          or RUN_PERSISTENCE or RUN_FUSION or RUN_FUSION_JOINT
                                          or RUN_RENDER_PREPARE or RUN_RENDER_VIDEO or RENDER_STORY
                                          or RENDER_WORLD):
    raise ValueError("with RUN_SWEEP_EVAL the single-run flags need PREDICTIONS_DIR set to the run to evaluate")

PROJECT_DIR = "/kaggle/working/Pointdit-In-Motion"
FORK_DIR = f"{PROJECT_DIR}/third_party/pointdit"
SEQUENCES_CONFIG = f"{PROJECT_DIR}/configs/sequences.yaml"

import base64
import os
import subprocess
import sys

from kaggle_secrets import UserSecretsClient


def github_token() -> str | None:
    try:
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None


# Passed to git through env vars, so the token never shows in commands, output or .git/config.
token = github_token()
if token:
    basic_auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    os.environ["GIT_CONFIG_COUNT"] = "1"
    os.environ["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    os.environ["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {basic_auth}"
    del basic_auth
print("GitHub auth:", "token" if token else "none (public repo)")
del token

if os.path.isdir(f"{PROJECT_DIR}/.git"):
    !git -C {PROJECT_DIR} fetch -q origin {PROJECT_BRANCH}
    !git -C {PROJECT_DIR} checkout -q {PROJECT_BRANCH}
    !git -C {PROJECT_DIR} pull -q --ff-only origin {PROJECT_BRANCH}
else:
    !rm -rf {PROJECT_DIR}
    !git clone --recurse-submodules --branch {PROJECT_BRANCH} {PROJECT_REPO} {PROJECT_DIR}
if not os.path.isdir(f"{PROJECT_DIR}/.git"):
    raise RuntimeError(f"Cloning {PROJECT_REPO} failed; see the git output above.")
!git -C {PROJECT_DIR} submodule update --init --recursive
%cd {PROJECT_DIR}
!echo "project $(git -C {PROJECT_DIR} rev-parse HEAD)"
!echo "fork    $(git -C {FORK_DIR} rev-parse HEAD)"


def run(*cmd: object, allowed: tuple[int, ...] = (0,)) -> int:
    """Run a project script, streaming its output, and fail the cell on an exit code outside allowed."""
    proc = subprocess.Popen([sys.executable, *map(str, cmd)], cwd=PROJECT_DIR, text=True,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() not in allowed:
        raise RuntimeError(f"{cmd[0]} exited with code {proc.returncode}; see the output above")
    return proc.returncode

## Dependencies

Installs the CPU requirements and prints the torch build.

In [ ]:
!pip install -q "numpy>=1.26.0" "Pillow>=10.0.0" "matplotlib>=3.9.0" "PyYAML>=6.0" "torch>=2.4"

import torch

print("torch", torch.__version__, "| CPUs", os.cpu_count(), "| CUDA", torch.cuda.is_available())

## Locate data and predictions

Finds the sequences and the attached `run.json` files under `/kaggle/input`.

In [ ]:
import csv
import json
from pathlib import Path

sys.path.insert(0, f"{PROJECT_DIR}/src")
from pim.data.tum import find_sequences
from pim.runs import IGNORED_INPUT_PARTS
from pim.runs import find_input_files as find_files

INPUT = Path("/kaggle/input")


def find_input_files(name: str) -> list[Path]:
    """Every file called `name` under /kaggle/input, following symlinked folders and ignoring any path that contains
    one of IGNORED_INPUT_PARTS, such as a project checkout bundled into an attached output."""
    return find_files(INPUT, name, IGNORED_INPUT_PARTS)


def input_overview(max_depth: int = 4) -> str:
    lines = []
    for root, dirs, files in os.walk(INPUT, followlinks=True):
        depth = len(Path(root).relative_to(INPUT).parts)
        if depth >= max_depth:
            dirs.clear()
        dirs[:] = [d for d in sorted(dirs) if d not in ("rgb", "depth")]
        lines.append(f"{'  ' * depth}{Path(root).name}/  ({len(files)} files)")
    return "\n".join(lines)


sequences = sorted(find_sequences("/kaggle/input"), key=lambda s: s.name)
for sequence in sequences:
    print(sequence)
data_roots = {s.parent for s in sequences}
if len(data_roots) != 1:
    raise RuntimeError(f"sequences are spread over several folders: {sorted(data_roots)}")
DATA_ROOT = data_roots.pop()

SAMPLING_RUN_NAMES = ("noise_seed0", "noise_seed1", "noise_seed2", "zeros_rerun", "baseline")


def find_runs(names: tuple[str, ...]) -> dict[str, Path]:
    by_name: dict[str, list[Path]] = {}
    for path in find_input_files("run.json"):
        by_name.setdefault(json.loads(path.read_text())["name"], []).append(path.parent)
    wrong = {name: [str(p) for p in by_name.get(name, [])] for name in names if len(by_name.get(name, [])) != 1}
    if wrong:
        raise RuntimeError(f"expected exactly one run.json per run under /kaggle/input; found {wrong}. Attach the runner "
                           "output with the sampling predictions and the one with predictions/baseline. "
                           f"/kaggle/input holds:\n{input_overview()}")
    return {name: by_name[name][0] for name in names}


if RUN_SAMPLING_EVAL:
    SAMPLING_RUNS = find_runs(SAMPLING_RUN_NAMES)
    for name, run_dir in SAMPLING_RUNS.items():
        record = json.loads((run_dir / "run.json").read_text())
        print(f"{name}: {run_dir} ({record['config']['inference']}, subsets {record.get('subsets')})")
if SINGLE_RUN:
    if PREDICTIONS_DIR is None:
        runs = find_input_files("run.json")
        if len(runs) != 1:
            raise RuntimeError(f"expected one run.json under /kaggle/input, found {[str(r) for r in runs]}. Attach the "
                               "output of the runner notebook version that wrote predictions/<run name>/run.json, or set "
                               f"PREDICTIONS_DIR in the first cell. /kaggle/input holds:\n{input_overview()}")
        RUN_DIR = runs[0].parent
    else:
        RUN_DIR = Path(PREDICTIONS_DIR)
    run_record = json.loads((RUN_DIR / "run.json").read_text())
    RUN_NAME = run_record["name"]
    EVAL_DIR = Path("/kaggle/working/eval") / RUN_NAME
    CONSISTENCY_DIR = Path("/kaggle/working/consistency") / RUN_NAME
    print(f"\npredictions {RUN_DIR} (run {RUN_NAME!r}, {run_record.get('start_time')} -> {run_record.get('end_time')})")
    for name, seq in run_record["sequences"].items():
        print(f"  {name}: window {seq['window']}, {len(list((RUN_DIR / name).glob('*.npz')))} npz files")
    print(f"results -> {EVAL_DIR}")


## Sampler initialisation comparison

Runs with `RUN_SAMPLING_EVAL` on the noise seed, zero rerun and baseline runs. It stops if the zero init pairs differ from `results/consistency/baseline` by more than 1e-3.

In [ ]:
SAMPLING_DIR = Path("/kaggle/working/sampling")
SUBSETS = f"{PROJECT_DIR}/configs/sampling_subsets.yaml"
SAMPLING_CONFIGS = ("zeros_rerun", "noise_seed0", "noise_seed1", "noise_seed2")


def run_parallel(*commands: list[object]) -> None:
    """Run project scripts concurrently, prefixing each output line with its index; fail if any fails."""
    import threading

    procs = [subprocess.Popen([sys.executable, *map(str, cmd)], cwd=PROJECT_DIR, text=True,
                              env={**os.environ, "PYTHONUNBUFFERED": "1"},
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT) for cmd in commands]

    def relay(k: int, proc: subprocess.Popen) -> None:
        for line in proc.stdout:
            print(f"[{k}] {line}", end="", flush=True)

    threads = [threading.Thread(target=relay, args=(k, p), daemon=True) for k, p in enumerate(procs)]
    for thread in threads:
        thread.start()
    codes = [p.wait() for p in procs]
    for thread in threads:
        thread.join()
    if any(codes):
        raise RuntimeError(f"exit codes {codes} for {[str(c[0]) for c in commands]}; see the output above")


if RUN_SAMPLING_EVAL:
    common = ["--data-root", DATA_ROOT, "--sequences-config", SEQUENCES_CONFIG, "--subsets", SUBSETS]
    run(f"{PROJECT_DIR}/scripts/check_determinism.py", "--predictions", SAMPLING_RUNS["zeros_rerun"],
        "--reference", SAMPLING_RUNS["baseline"], "--subsets", SUBSETS, "--out", SAMPLING_DIR / "determinism.csv")
    for name in SAMPLING_CONFIGS:
        run(f"{PROJECT_DIR}/scripts/evaluate.py", "--predictions", SAMPLING_RUNS[name], *common,
            "--out", SAMPLING_DIR / "eval" / name, "--workers", 4)
    # One worker per sequence and two sequences per subset, so two configs at a time fill 4 CPUs.
    for batch in (SAMPLING_CONFIGS[:2], SAMPLING_CONFIGS[2:]):
        run_parallel(*[[f"{PROJECT_DIR}/scripts/consistency_eval.py", "--predictions", SAMPLING_RUNS[name], *common,
                        "--out", SAMPLING_DIR / "consistency" / name, "--workers", 2, "--threads", 1]
                       for name in batch])
    run(f"{PROJECT_DIR}/scripts/sampling_compare.py", "--noise", *(SAMPLING_RUNS[n] for n in SAMPLING_CONFIGS[1:]),
        "--zeros", SAMPLING_RUNS["zeros_rerun"], "--eval-root", SAMPLING_DIR / "eval",
        "--consistency-root", SAMPLING_DIR / "consistency",
        "--baseline-consistency", f"{PROJECT_DIR}/results/consistency/baseline", *common,
        "--out", SAMPLING_DIR, "--workers", 4, "--threads", 1)
    print((SAMPLING_DIR / "summary.md").read_text())
    print("to download:")
    for path in sorted(SAMPLING_DIR.rglob("*")):
        if path.is_file():
            print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")
else:
    print("RUN_SAMPLING_EVAL is False; skipping the sampling comparison")


## Compute sweep evaluation

Runs with `RUN_SWEEP_EVAL` on every sweep run whose status is ok. A failed baseline regression check only prints a warning.

In [ ]:
SWEEP_EVAL_DIR = Path("/kaggle/working/sweep")
SWEEP_SUBSETS = f"{PROJECT_DIR}/configs/sweep_subsets.yaml"
SWEEP_CHECK = "sweep_baseline_check"
# sweep_accuracy.py exit code when only the final regression check fails.
REGRESSION_MISMATCH = 3


def find_sweep_runs() -> dict[str, Path]:
    """sweep_* run folders under /kaggle/input by run.json, skipping copies inside an attached project clone."""
    by_name: dict[str, list[Path]] = {}
    for path in find_input_files("run.json"):
        if "/Pointdit-In-Motion/" in str(path):
            continue
        name = json.loads(path.read_text()).get("name", "")
        if name.startswith("sweep_"):
            by_name.setdefault(name, []).append(path.parent)
    several = {name: [str(p) for p in paths] for name, paths in by_name.items() if len(paths) > 1}
    if several or not by_name:
        raise RuntimeError(f"expected one run.json per sweep run under /kaggle/input, found {several or 'none'}. "
                           f"Attach the runner output with predictions/sweep. /kaggle/input holds:\n{input_overview()}")
    return {name: paths[0] for name, paths in sorted(by_name.items())}


def sweep_status(run_dir: Path) -> str:
    path = run_dir / "sweep_status.json"
    return json.loads(path.read_text()).get("status", "unknown") if path.is_file() else "no sweep_status.json"


def run_code(*cmd: object) -> int:
    """Like run(), but returns the exit code instead of raising."""
    proc = subprocess.Popen([sys.executable, *map(str, cmd)], cwd=PROJECT_DIR, text=True,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()


if RUN_SWEEP_EVAL:
    sweep_runs = find_sweep_runs()
    statuses = {name: sweep_status(run_dir) for name, run_dir in sweep_runs.items()}
    for name, run_dir in sweep_runs.items():
        print(f"{name}: {statuses[name]} ({run_dir})")
    succeeded = [name for name, status in statuses.items() if status == "ok"]
    if SWEEP_CHECK not in succeeded:
        raise RuntimeError(f"{SWEEP_CHECK} is not among the successful sweep runs {succeeded}; it is the reference")
    # Evaluate every config first, then raise for the failed ones.
    failed = {}
    for name in succeeded:
        code = run_code(f"{PROJECT_DIR}/scripts/evaluate.py", "--predictions", sweep_runs[name], "--data-root", DATA_ROOT,
                        "--sequences-config", SEQUENCES_CONFIG, "--subsets", SWEEP_SUBSETS,
                        "--out", SWEEP_EVAL_DIR / "eval" / name, "--workers", 4)
        if code:
            failed[name] = code
    code = run_code(f"{PROJECT_DIR}/scripts/sweep_accuracy.py", "--eval-root", SWEEP_EVAL_DIR / "eval",
                    "--configs", *(n for n in succeeded if n not in failed), "--subsets", SWEEP_SUBSETS,
                    "--baseline-csv", f"{PROJECT_DIR}/results/eval/baseline/per_frame.csv",
                    "--out", SWEEP_EVAL_DIR / "sweep_accuracy.csv")
    print("to download:")
    for path in sorted(p for p in SWEEP_EVAL_DIR.rglob("*") if p.is_file()):
        print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")
    if code == REGRESSION_MISMATCH:
        print(f"\nWARNING the regression check of {SWEEP_CHECK} failed (see the banner above); outputs are kept")
    elif code:
        raise RuntimeError(f"sweep_accuracy.py exited with code {code}; see the output above")
    if failed:
        raise RuntimeError(f"evaluate.py failed for {failed} (run name and exit code); the other configs are tabulated")
else:
    print("RUN_SWEEP_EVAL is False; skipping the sweep evaluation")

## Temporal consistency

Runs with `RUN_CONSISTENCY` and stops if any subtask 1 value differs by more than 1e-3. Complete sequences are skipped on a rerun.

In [ ]:
if RUN_CONSISTENCY:
    run(f"{PROJECT_DIR}/scripts/consistency_eval.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
        "--sequences-config", SEQUENCES_CONFIG, "--out", CONSISTENCY_DIR, "--workers", 4, "--threads", 1,
        "--save-maps", "rgbd_dataset_freiburg1_360", "rgbd_dataset_freiburg1_desk")
else:
    print("RUN_CONSISTENCY is False; skipping consistency_eval.py")


## Error persistence

Runs with `RUN_PERSISTENCE` and stops if any pair differs from the bundled consistency results by more than 1e-6.

In [ ]:
PERSISTENCE_DIR = Path("/kaggle/working/persistence")
if RUN_PERSISTENCE:
    run(f"{PROJECT_DIR}/scripts/persistence_eval.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
        "--sequences-config", SEQUENCES_CONFIG, "--out", PERSISTENCE_DIR, "--workers", 4, "--threads", 1,
        "--reference", f"{PROJECT_DIR}/results/consistency/baseline")
    print("to download:")
    for path in sorted(PERSISTENCE_DIR.glob("*")):
        print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")
else:
    print("RUN_PERSISTENCE is False; skipping persistence_eval.py")


## Per-frame results

Reuses the one attached `per_frame.csv`, or reruns `evaluate.py` with `RUN_EVALUATION` and checks it against an attached older CSV within 1e-6.

In [ ]:
if not SINGLE_RUN:
    print("RUN_SAMPLING_EVAL or RUN_SWEEP_EVAL without PREDICTIONS_DIR; skipping the single-run steps")
else:
    def attached_per_frame_csvs() -> list[Path]:
        """per_frame.csv files of attached inputs only, or exactly PER_FRAME_CSV_OVERRIDE when it is set; never one
        this session wrote to /kaggle/working."""
        working = Path("/kaggle/working").resolve()
        if PER_FRAME_CSV_OVERRIDE is not None:
            override = Path(PER_FRAME_CSV_OVERRIDE)
            if not override.is_file():
                raise FileNotFoundError(f"PER_FRAME_CSV_OVERRIDE {override} does not exist")
            print(f"using PER_FRAME_CSV_OVERRIDE {override}")
            found = [override]
        else:
            found = find_input_files("per_frame.csv")
        written_here = [p for p in found if p.resolve().is_relative_to(working)]
        if written_here:
            raise RuntimeError(f"{[str(p) for p in written_here]} under /kaggle/input resolve into /kaggle/working")
        return found


    def check_evaluates_this_run(csv_path: Path) -> None:
        eval_json = csv_path.parent / "eval.json"
        if not eval_json.is_file():
            raise RuntimeError(f"{eval_json} not found next to {csv_path}; cannot tell which run it evaluated")
        evaluated_run = json.loads(eval_json.read_text())["predictions"]["run"]
        if evaluated_run != run_record:
            raise RuntimeError(f"{csv_path} evaluates run {evaluated_run.get('name')!r} started "
                               f"{evaluated_run.get('start_time')}, but the attached predictions are run {RUN_NAME!r} "
                               f"started {run_record.get('start_time')}")


    # Search before evaluating so the new CSV is never a candidate.
    attached = attached_per_frame_csvs()
    if RUN_EVALUATION:
        if len(attached) > 1:
            raise RuntimeError(f"found several per_frame.csv under /kaggle/input: {[str(p) for p in attached]}; "
                               "attach at most one older evaluation to compare against")
        for old_csv in attached:
            check_evaluates_this_run(old_csv)
        run(f"{PROJECT_DIR}/scripts/evaluate.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
            "--sequences-config", SEQUENCES_CONFIG, "--out", EVAL_DIR, "--workers", WORKERS)
        CSV_PATH = EVAL_DIR / "per_frame.csv"
        if attached:
            print(f"\ncomparing {CSV_PATH} with {attached[0]}")
            run(f"{PROJECT_DIR}/scripts/compare_eval_csv.py", CSV_PATH, attached[0])
        else:
            print("\nno older per_frame.csv attached under /kaggle/input; skipping the reproduction check")
    else:
        if len(attached) != 1:
            raise RuntimeError(f"RUN_EVALUATION is False, so exactly one per_frame.csv must be attached under /kaggle/input; "
                               f"found {[str(p) for p in attached]}. Attach the output of the evaluation notebook version "
                               "that evaluated this run, or set RUN_EVALUATION = True.")
        CSV_PATH = attached[0]
        check_evaluates_this_run(CSV_PATH)
    print(f"per-frame results {CSV_PATH}")


## Focal split

Runs with `RUN_FOCAL_SPLIT` and refits the implied focal length on all pixels and on each image side.

In [ ]:
if RUN_FOCAL_SPLIT:
    run(f"{PROJECT_DIR}/scripts/focal_reliability.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
        "--sequences-config", SEQUENCES_CONFIG, "--csv", CSV_PATH, "--out", EVAL_DIR / "focal_split.csv")
else:
    print("RUN_FOCAL_SPLIT is False; skipping focal_reliability.py")


## Oracle scale fusion

Runs with `RUN_FUSION`, aligning every frame to its own GT first, so results are an upper bound. A preflight stops on a mismatch, while the final `baseline_self` check only warns.

In [ ]:
FUSION_DIR = Path("/kaggle/working/fusion")
# "ladder" is the original grid, "edge" the edge-aware configs written to oracle_eval_edge/.
FUSION_CONFIG_GROUP = "ladder"
# None aligns every frame (~1 h); a glob like "/kaggle/input/*/fusion/oracle_aligned" reuses one cache.
FUSION_CACHE_INPUT = None
if RUN_FUSION:
    import glob
    import shutil

    # ORACLE scale, every source aligned to its own GT, so results are an upper bound.
    FUSION_CACHE = FUSION_DIR / "oracle_aligned" / RUN_NAME
    eval_folder = "oracle_eval" if FUSION_CONFIG_GROUP == "ladder" else f"oracle_eval_{FUSION_CONFIG_GROUP}"
    FUSION_OUT = FUSION_DIR / eval_folder / RUN_NAME
    if FUSION_CACHE_INPUT:
        matches = sorted(glob.glob(FUSION_CACHE_INPUT))
        assert len(matches) == 1, f"FUSION_CACHE_INPUT {FUSION_CACHE_INPUT} must match one path, got {matches}"
        cache_input = Path(matches[0])
        if cache_input.name != "oracle_aligned":
            cache_input = cache_input / "fusion" / "oracle_aligned"
        assert (cache_input / RUN_NAME / "oracle_prepare.json").is_file(), f"{cache_input} has no {RUN_NAME} cache"
        print(f"copying {cache_input} to {FUSION_DIR / 'oracle_aligned'}")
        shutil.copytree(cache_input, FUSION_DIR / "oracle_aligned", dirs_exist_ok=True,
                        copy_function=shutil.copyfile)
    fusion_common = ["--data-root", DATA_ROOT, "--sequences-config", SEQUENCES_CONFIG]
    run(f"{PROJECT_DIR}/scripts/fusion_prepare.py", "--predictions", RUN_DIR, *fusion_common, "--out", FUSION_CACHE,
        "--workers", 4, "--threads", 1)
    fusion_eval_args = ["--cache", FUSION_CACHE, *fusion_common, "--baseline-csv", CSV_PATH, "--workers", 4,
                        "--threads", 1, "--config-group", FUSION_CONFIG_GROUP]
    # None evaluates every sequence.
    FUSION_SEQUENCES = None
    fusion_sequences = ["--sequences", *FUSION_SEQUENCES] if FUSION_SEQUENCES else []
    fusion_eval = f"{PROJECT_DIR}/scripts/fusion_eval.py"
    run(fusion_eval, *fusion_eval_args, *fusion_sequences, "--out", FUSION_OUT, "--preflight")
    # Exit code 3 means outputs are written but baseline_self differs from per_frame.csv.
    if run(fusion_eval, *fusion_eval_args, *fusion_sequences, "--out", FUSION_OUT, allowed=(0, 3)) == 3:
        print("WARNING fusion_eval.py baseline_self regression check FAILED; outputs are kept, see below")
    print((FUSION_OUT / "regression_status.json").read_text())
    if FUSION_ALL_TARGETS_CONFIG:
        run(fusion_eval, *fusion_eval_args, "--all-targets",
            "--configs", FUSION_ALL_TARGETS_CONFIG,
            "--out", FUSION_DIR / f"{eval_folder}_all_targets" / RUN_NAME)
    print("to download (the oracle_aligned cache is not needed):")
    for path in sorted(FUSION_DIR.rglob("*")):
        if path.is_file() and path.relative_to(FUSION_DIR).parts[0] != "oracle_aligned":
            print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")
    print(f"  {FUSION_CACHE / 'oracle_prepare.json'} (provenance of the cache)")
else:
    print("RUN_FUSION is False; skipping fusion_prepare.py and fusion_eval.py")


## Joint scale fusion

Runs with `RUN_FUSION_JOINT`, solving scale from parallax under oracle GT poses. It needs the aligned depth cache from `RUN_FUSION` or `FUSION_CACHE_INPUT`.

In [ ]:
if RUN_FUSION_JOINT:
    import glob

    from pim.fusion.oracle import resolve_cache_dir

    # Scale joint_solved, poses oracle_gt; the GT-aligned cache is read only for logging.
    if RUN_FUSION:
        joint_cache = FUSION_DIR / "oracle_aligned" / RUN_NAME
    else:
        assert FUSION_CACHE_INPUT, "RUN_FUSION_JOINT needs FUSION_CACHE_INPUT (or RUN_FUSION = True) for the cache"
        matches = sorted(glob.glob(FUSION_CACHE_INPUT))
        assert len(matches) == 1, f"FUSION_CACHE_INPUT {FUSION_CACHE_INPUT} must match one path, got {matches}"
        # Accepts the output root, its oracle_aligned folder or the <run name> cache itself.
        joint_cache = resolve_cache_dir(Path(matches[0]), RUN_NAME)
    assert (joint_cache / "oracle_prepare.json").is_file(), f"{joint_cache} has no oracle_prepare.json"
    # This session's oracle edge summary if present, else the bundled one.
    joint_oracle_summary = FUSION_DIR / "oracle_eval_edge" / RUN_NAME / "oracle_fusion_summary.csv"
    if not joint_oracle_summary.is_file():
        joint_oracle_summary = Path(PROJECT_DIR) / "results" / "fusion" / "oracle_eval_edge" / RUN_NAME / \
            "oracle_fusion_summary.csv"
    JOINT_OUT = FUSION_DIR / "joint_eval" / RUN_NAME
    # None evaluates every sequence of the cache.
    JOINT_SEQUENCES = None
    joint_args = ["--predictions", RUN_DIR, "--cache", joint_cache, "--data-root", DATA_ROOT,
                  "--sequences-config", SEQUENCES_CONFIG, "--baseline-csv", CSV_PATH,
                  "--oracle-summary", joint_oracle_summary, "--workers", 4, "--threads", 1, "--out", JOINT_OUT,
                  *(["--sequences", *JOINT_SEQUENCES] if JOINT_SEQUENCES else [])]
    fusion_joint = f"{PROJECT_DIR}/scripts/fusion_joint.py"
    run(fusion_joint, *joint_args, "--preflight")
    # Exit code 3 means outputs are written but baseline_self or the no-fusion control differs.
    if run(fusion_joint, *joint_args, allowed=(0, 3)) == 3:
        print("WARNING fusion_joint.py regression check FAILED; outputs are kept, see below")
    print((JOINT_OUT / "regression_status.json").read_text())
    print("to download")
    for path in sorted(JOINT_OUT.rglob("*")):
        if path.is_file():
            print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")
else:
    print("RUN_FUSION_JOINT is False; skipping fusion_joint.py")


## Render preparation

Runs with `RUN_RENDER_PREPARE` and saves raw and fused depth of every window frame by `joint_causal_adjacent_w4`. Frames already written are skipped on a rerun.

In [ ]:
if RUN_RENDER_PREPARE:
    import glob

    from pim.fusion.oracle import resolve_cache_dir

    # Scale joint_solved, poses oracle_gt; the cache is read only for provenance and windows.
    if RUN_FUSION:
        render_cache = FUSION_DIR / "oracle_aligned" / RUN_NAME
    else:
        assert FUSION_CACHE_INPUT, "RUN_RENDER_PREPARE needs FUSION_CACHE_INPUT (or RUN_FUSION = True) for the cache"
        matches = sorted(glob.glob(FUSION_CACHE_INPUT))
        assert len(matches) == 1, f"FUSION_CACHE_INPUT {FUSION_CACHE_INPUT} must match one path, got {matches}"
        render_cache = resolve_cache_dir(Path(matches[0]), RUN_NAME)
    # This session's joint results if present, else the bundled ones.
    render_joint = FUSION_DIR / "joint_eval" / RUN_NAME
    if not (render_joint / "regression_status.json").is_file():
        render_joint = Path(PROJECT_DIR) / "results" / "fusion" / "joint_eval" / RUN_NAME
    RENDER_DIR = FUSION_DIR / "render"
    RENDER_OUT = RENDER_DIR / RUN_NAME
    render_args = ["--predictions", RUN_DIR, "--cache", render_cache, "--data-root", DATA_ROOT,
                   "--sequences-config", SEQUENCES_CONFIG, "--baseline-csv", CSV_PATH, "--joint-csv", render_joint,
                   "--workers", 4, "--threads", 1, "--out", RENDER_DIR,
                   *(["--sequences", *RENDER_SEQUENCES] if RENDER_SEQUENCES else [])]
    render_prepare = f"{PROJECT_DIR}/scripts/fusion_render_prepare.py"
    run(render_prepare, *render_args, "--preflight")
    # Exit code 3 means outputs are written but rel_d_raw or rel_d_fused differs.
    if run(render_prepare, *render_args, allowed=(0, 3)) == 3:
        print("WARNING fusion_render_prepare.py regression check FAILED; outputs are kept, see below")
    print((RENDER_OUT / "regression_status.json").read_text())
    print("to download")
    for path in sorted(RENDER_OUT.glob("*.json")):
        print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")
    for folder in sorted(p for p in RENDER_OUT.iterdir() if p.is_dir()):
        files = list(folder.glob("*.npz"))
        print(f"  {folder}/ ({len(files)} npz, {sum(p.stat().st_size for p in files) / 1e6:.0f} MB)")
else:
    print("RUN_RENDER_PREPARE is False; skipping fusion_render_prepare.py")


## Render input

Picks the render preparation output for the video cells, from this session or `RENDER_INPUT`.

In [ ]:
VIDEO_DIR = Path("/kaggle/working/videos")
if RUN_RENDER_VIDEO or RENDER_STORY or RENDER_WORLD:
    import glob

    def render_frames_dir(path: Path) -> Path:
        """The folder holding the sequence folders of npz frames, given it or a notebook output root."""
        if any(next(p.glob("*.npz"), None) for p in path.iterdir() if p.is_dir()):
            return path
        candidate = path / "fusion" / "render" / RUN_NAME
        if candidate.is_dir():
            return candidate
        raise FileNotFoundError(f"no sequence folders of npz frames in {path} or in {candidate}")

    if RENDER_INPUT is None:
        assert RUN_RENDER_PREPARE, ("RUN_RENDER_VIDEO, RENDER_STORY and RENDER_WORLD need RENDER_INPUT (or "
                                    "RUN_RENDER_PREPARE = True)")
        video_input = RENDER_OUT
    else:
        matches = sorted(glob.glob(str(RENDER_INPUT)))
        assert len(matches) == 1, f"RENDER_INPUT {RENDER_INPUT} must match one path, got {matches}"
        video_input = render_frames_dir(Path(matches[0]))
    video_sequences = RENDER_SEQUENCES or sorted(p.name for p in video_input.iterdir() if p.is_dir())
    print(f"render input {video_input}, sequences {video_sequences}")

## Dashboard videos and point maps

Runs with `RUN_RENDER_VIDEO`, writing to `/kaggle/working/videos/`. Without ffmpeg the videos are PNG frame folders.

In [ ]:
if RUN_RENDER_VIDEO:
    # Scale joint_solved, poses oracle_gt, an ORACLE upper bound. CPU only, one process at a time.
    for name in video_sequences:
        run(f"{PROJECT_DIR}/scripts/render_video.py", video_input, DATA_ROOT, SEQUENCES_CONFIG, name, VIDEO_DIR)
    map_name = video_sequences[0]
    MAP_DIR = VIDEO_DIR / f"{map_name}_map"
    run(f"{PROJECT_DIR}/scripts/export_fused_map.py", video_input, DATA_ROOT, SEQUENCES_CONFIG, map_name, MAP_DIR)
    run(f"{PROJECT_DIR}/scripts/render_orbit.py", MAP_DIR, VIDEO_DIR / f"{map_name}_orbit")
    if RENDER_FLYTHROUGH:
        run(f"{PROJECT_DIR}/scripts/render_orbit.py", MAP_DIR, VIDEO_DIR / f"{map_name}_orbit", "--flythrough")
    print(json.dumps(json.loads((MAP_DIR / "map.json").read_text())["ghosting"], indent=2))
    print("to download")
    frame_folders: dict[Path, list[Path]] = {}
    for path in sorted(VIDEO_DIR.rglob("*")):
        if not path.is_file():
            continue
        if path.suffix == ".png" and path.stem.isdigit():
            frame_folders.setdefault(path.parent, []).append(path)
        else:
            print(f"  {path} ({path.stat().st_size / 1e6:.1f} MB)")
    for folder, files in frame_folders.items():
        print(f"  {folder}/ ({len(files)} PNG frames without ffmpeg, {sum(p.stat().st_size for p in files) / 1e6:.0f} MB)")
else:
    print("RUN_RENDER_VIDEO is False; skipping render_video.py, export_fused_map.py and render_orbit.py")


## Story clips

Runs with `RENDER_STORY`, writing short slower clips to `/kaggle/working/videos/story/`.

In [ ]:
if RENDER_STORY:
    # Scale joint_solved, poses oracle_gt, an ORACLE upper bound. CPU only, one process at a time.
    STORY_DIR = VIDEO_DIR / "story"
    for name in video_sequences:
        for mode in STORY_MODES:
            run(f"{PROJECT_DIR}/scripts/render_story.py", video_input, DATA_ROOT, SEQUENCES_CONFIG, name, STORY_DIR,
                "--mode", mode, *STORY_ARGS.get(mode, []))
    map_name = video_sequences[0]
    MAP_DIR = VIDEO_DIR / f"{map_name}_map"
    if not (MAP_DIR / "map.json").is_file():
        run(f"{PROJECT_DIR}/scripts/export_fused_map.py", video_input, DATA_ROOT, SEQUENCES_CONFIG, map_name, MAP_DIR)
    run(f"{PROJECT_DIR}/scripts/render_orbit.py", MAP_DIR, STORY_DIR / f"{map_name}_orbit", "--side-by-side",
        "--fps", 10, "--seconds", 36, "--hold-seconds", 1)
    for path in sorted(STORY_DIR.glob("*.json")):
        record = json.loads(path.read_text())
        medians = record["clip_medians"]
        frame = record.get("frame", {}).get("index")
        print(f"{path.name}  frames {record['clip']['frames']}  disagreement raw "
              f"{medians['median_disagreement_raw_pct']} fused {medians['median_disagreement_fused_pct']} %"
              + (f"  frame {frame} ({record['frame']['chosen_by']})" if frame is not None else ""))
    print("to download")
    for path in sorted(STORY_DIR.rglob("*")):
        if path.is_file() and not (path.suffix == ".png" and path.stem.isdigit()):
            print(f"  {path} ({path.stat().st_size / 1e6:.1f} MB)")
    for folder in sorted({p.parent for p in STORY_DIR.rglob("[0-9]*.png")}):
        print(f"  {folder}/ (PNG frames without ffmpeg)")
else:
    print("RENDER_STORY is False; skipping render_story.py and the slowed orbit")

## World frame videos

Runs with `RENDER_WORLD`, writing raw and fused world frame point maps to `/kaggle/working/videos/world/`.

In [ ]:
if RENDER_WORLD:
    # Scale joint_solved, poses oracle_gt, an ORACLE upper bound, not 4D. CPU only, one process at a time.
    WORLD_DIR = VIDEO_DIR / "world"
    for name in video_sequences:
        for mode in WORLD_MODES:
            run(f"{PROJECT_DIR}/scripts/render_world_video.py", video_input, DATA_ROOT, SEQUENCES_CONFIG, name,
                WORLD_DIR, "--mode", mode, *WORLD_ARGS)
    for path in sorted(WORLD_DIR.glob("*.json")):
        record = json.loads(path.read_text())
        ghost = {s: g["std_m"] for s, g in record["ghosting"].items()}
        print(f"{path.name}  frames {len(record['frames_used'])}  final points {record['points']['final_shown']['raw']}"
              f"  ghosting std m {ghost}  peak RSS {record['peak_rss_mb']} MB")
    print("to download")
    for path in sorted(WORLD_DIR.rglob("*")):
        if path.is_file() and not (path.suffix == ".png" and path.stem.isdigit()):
            print(f"  {path} ({path.stat().st_size / 1e6:.1f} MB)")
    for folder in sorted({p.parent for p in WORLD_DIR.rglob("[0-9]*.png")}):
        print(f"  {folder}/ (PNG frames without ffmpeg)")
else:
    print("RENDER_WORLD is False; skipping render_world_video.py")

## Frame checks

Runs `check_frame.py` on the middle frame of each evaluated sequence.

In [ ]:
if not SINGLE_RUN:
    print("RUN_SAMPLING_EVAL or RUN_SWEEP_EVAL without PREDICTIONS_DIR; skipping the single-run steps")
else:
    with open(CSV_PATH, newline="") as f:
        rows = list(csv.DictReader(f))
    evaluated = list(dict.fromkeys(r["sequence"] for r in rows))

    for name in evaluated:
        window = run_record["sequences"][name]["window"]
        middle = window["start"] + window["count"] // 2
        run(f"{PROJECT_DIR}/scripts/check_frame.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
            "--sequences-config", SEQUENCES_CONFIG, "--csv", CSV_PATH, "--sequence", name, "--frame", middle)
        print()


## Previews

Writes RGB, predicted depth, GT depth and error panels for the worst, median and selected room frames.

In [ ]:
if not SINGLE_RUN:
    print("RUN_SAMPLING_EVAL or RUN_SWEEP_EVAL without PREDICTIONS_DIR; skipping the single-run steps")
else:
    import numpy as np

    WORST = 5
    ROOM = "rgbd_dataset_freiburg1_room"
    ROOM_FRAMES = range(366, 381)
    PREVIEW_DIR = Path("/kaggle/working/previews") / RUN_NAME
    BREAKDOWN_CSV = PREVIEW_DIR / "error_breakdown.csv"

    if ROOM not in evaluated:
        raise RuntimeError(f"{CSV_PATH} has no {ROOM} rows; evaluated sequences are {evaluated}")


    def preview_frames(name: str) -> dict[int, str]:
        """Frame index -> why it is previewed."""
        scored = sorted((r for r in rows if r["sequence"] == name and r["rel_d"]), key=lambda r: float(r["rel_d"]))
        if not scored:
            raise RuntimeError(f"{name}: no frame with a rel_d in {CSV_PATH}")
        median = float(np.median([float(r["rel_d"]) for r in scored]))
        closest = min(scored, key=lambda r: abs(float(r["rel_d"]) - median))
        reasons: dict[int, list[str]] = {}
        for rank, r in enumerate(reversed(scored[-WORST:]), start=1):
            reasons.setdefault(int(r["frame_index"]), []).append(f"worst{rank}")
        reasons.setdefault(int(closest["frame_index"]), []).append("median")
        if name == ROOM:
            for index in ROOM_FRAMES:
                reasons.setdefault(index, []).append("room366-380")
        return {index: "+".join(why) for index, why in sorted(reasons.items())}


    # The script merges into an existing CSV, so start clean.
    BREAKDOWN_CSV.unlink(missing_ok=True)
    reasons = {}
    for name in evaluated:
        frames = preview_frames(name)
        reasons.update({(name, index): why for index, why in frames.items()})
        print(name, frames)
        run(f"{PROJECT_DIR}/scripts/preview_predictions.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
            "--sequences-config", SEQUENCES_CONFIG, "--sequence", name, "--frames", *frames,
            "--csv", CSV_PATH, "--out", PREVIEW_DIR)
        print()


## Error breakdown and downloads

Prints the edge band breakdown per previewed frame and lists the files to download.

In [ ]:
if not SINGLE_RUN:
    print("RUN_SAMPLING_EVAL or RUN_SWEEP_EVAL without PREDICTIONS_DIR; skipping the single-run steps")
else:
    with open(BREAKDOWN_CSV, newline="") as f:
        breakdown = list(csv.DictReader(f))


    def number(value: str, digits: int) -> str:
        return f"{float(value):.{digits}f}" if value else "n/a"


    print("band_px / band_err: share of valid pixels / of total relative error within 5 px of a GT depth edge or hole.")
    print("band_err >> band_px: error sits on edges; band_err ~ band_px with high rel_out: error spread over regions.")
    for name in evaluated:
        print(f"\n{name}")
        print(f"{'frame':>6} {'rel_d':>7} {'band_px':>8} {'band_err':>8} {'ratio':>6} {'rel_band':>8} {'rel_out':>8} "
              f"{'offset_ms':>9}  why")
        for r in (r for r in breakdown if r["sequence"] == name):
            ratio = (f"{float(r['band_error_fraction']) / float(r['band_pixel_fraction']):.2f}"
                     if r["band_error_fraction"] and float(r["band_pixel_fraction"]) > 0 else "n/a")
            print(f"{r['frame_index']:>6} {number(r['rel_d'], 2):>7} {number(r['band_pixel_fraction'], 3):>8} "
                  f"{number(r['band_error_fraction'], 3):>8} {ratio:>6} {number(r['rel_d_band'], 2):>8} "
                  f"{number(r['rel_d_outside'], 2):>8} {number(r['depth_offset_ms'], 1):>9}  "
                  f"{reasons[name, int(r['frame_index'])]}")

    print("\nto download:")
    for directory in (EVAL_DIR, PREVIEW_DIR, CONSISTENCY_DIR):
        for path in sorted(directory.rglob("*")) if directory.is_dir() else []:
            if path.is_file() and "maps" not in path.relative_to(directory).parts:
                print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")
    for maps in sorted((CONSISTENCY_DIR / "maps").glob("*")) if (CONSISTENCY_DIR / "maps").is_dir() else []:
        print(f"  {maps}/ ({len(list(maps.glob('*.npz')))} npz, "
              f"{sum(p.stat().st_size for p in maps.glob('*.npz')) / 1e6:.0f} MB)")
